# From weather to grid inputs

The same 2030 Spanish hour changes three inputs to grid operation: electricity
demand, available generation and transmission limits. These plots show the
modelled inputs and response relationships; no OPF is solved here.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "TDOPF_eur.py").is_file())
sys.path.insert(0, str(root))
data = root / "vis/demos/data/spain_2030"
demand = pd.read_csv(data / "demand_hourly.csv")
generators = pd.read_csv(data / "generator_inputs.csv")
lines = pd.read_csv(data / "line_inputs.csv")

## Demand

The calibrated demand model gives this heatwave day a daily demand level and
an hourly shape. The 1% annual growth assumption applies only to its base-demand
component. The 13:00 value is the active demand used by the grid example.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 3.2))
axis.plot(demand.hour, demand.modelled_demand_mw / 1000,
          color="#be4a3d", lw=2)
axis.scatter([13], [demand.loc[13, "modelled_demand_mw"] / 1000],
             color="#a22423", zorder=3)
axis.set(xlabel="Hour", ylabel="Modelled demand (GW)", xlim=(0, 23))
axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Generation

Ambient temperature reduces the modelled availability of some thermal
generators. Renewable availability is supplied by weather-dependent profiles.
The two effects are represented separately below.

In [ ]:
from utils.heat_flow_utils import generator_derating

temperatures = np.linspace(20, 45, 100)
fig, axis = plt.subplots(figsize=(7, 3.2))
for carrier, colour in [("CCGT", "#c44536"), ("nuclear", "#235a8c")]:
    curve = [generator_derating(carrier, {"air_temperature": t})
             for t in temperatures]
    axis.plot(temperatures, curve, lw=2, color=colour, label=carrier)
axis.set(xlabel="Air temperature at generator bus (C)",
         ylabel="Temperature derating factor", ylim=(0.7, 1.02))
axis.legend(frameon=False)
axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

summary = (generators.loc[generators.carrier.isin(["solar", "onwind", "CCGT", "nuclear"])]
           .groupby("carrier")[["p_max_pu", "derating", "bus_temperature_c"]]
           .median().round(2))
summary

For solar and wind, `p_max_pu` reflects the selected weather profile. For
CCGT and nuclear units, the separate `derating` factor describes the
temperature response. The table gives medians across represented units,
not national generation totals.

## Transmission

The model calculates a weather-dependent current bound along each represented
line. Some lines remain at the nominal bound, so a hotter line does not always
correspond to a lower displayed current limit.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))
points = axis.scatter(lines.air_temperature_c, lines.available_current_a,
                      c=lines.wind_speed_m_s, cmap="viridis", s=16, alpha=0.7)
axis.set(xlabel="Mean air temperature along line (C)",
         ylabel="Available current (A)")
axis.grid(alpha=0.2)
fig.colorbar(points, ax=axis, label="Mean 10-m wind speed (m/s)")
fig.tight_layout()
plt.show()

The current bound is an input to the OPF; the conductor temperature that
results from the solved flows is examined in the next notebook.